In [2]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

import joblib

In [3]:
df = pd.read_csv("../data/processed/cleaned_DDI_data.csv")

In [4]:
X = df[["drug1_name", "drug2_name"]]

y = df["interaction_type"]

In [5]:
X = df[["drug1_name", "drug2_name"]].copy()

drug_encoder = LabelEncoder()

all_drug_names = pd.concat([
    X["drug1_name"],
    X["drug2_name"]
])

drug_encoder.fit(all_drug_names)

X["drug1_name"] = drug_encoder.transform(X["drug1_name"])
X["drug2_name"] = drug_encoder.transform(X["drug2_name"])

X.head()

,drug1_name,drug2_name
0,205,42
1,205,6
2,205,106
3,205,450
4,205,601


In [8]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(df["interaction_type"])

y[:10]

array([101,  65,   6,   6,   6,   6, 101,  62, 101, 101])

In [9]:
class_counts = df["interaction_type"].value_counts()

rare_classes = class_counts[class_counts < 2]

print("Classes with fewer than 2 samples:")
print(rare_classes)

print("\nTotal rare classes:", len(rare_classes))

Classes with fewer than 2 samples:
interaction_type
thrombocytopenic activities                                    1
risk or severity of QTc prolongation and torsade de pointes    1
risk or severity of renal failure and rhabdomyolysis           1
risk or severity of pulmonary toxicity                         1
antineoplastic activities                                      1
hypotensive activities of at                                   1
risk or severity of generalized seizure and bradycardia        1
risk or severity of intraocular pressure                       1
serum concentration of sulfate                                 1
risk or severity of sinus node depression                      1
teratogenic activities                                         1
Name: count, dtype: int64

Total rare classes: 11


In [10]:
valid_classes = class_counts[class_counts >= 2].index

df = df[df["interaction_type"].isin(valid_classes)].copy()

In [11]:
X = df[["drug1_name", "drug2_name"]].copy()

drug_encoder = LabelEncoder()

all_drugs = pd.concat([
    X["drug1_name"],
    X["drug2_name"]
])

drug_encoder.fit(all_drugs)

X["drug1_name"] = drug_encoder.transform(X["drug1_name"])
X["drug2_name"] = drug_encoder.transform(X["drug2_name"])

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(df["interaction_type"])

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [13]:
print("Training samples :", X_train.shape)
print("Testing samples  :", X_test.shape)

print("Training labels  :", len(y_train))
print("Testing labels   :", len(y_test))

Training samples : (178108, 2)
Testing samples  : (44527, 2)
Training labels  : 178108
Testing labels   : 44527


In [14]:
import joblib

joblib.dump(drug_encoder, "../models/drug_encoder.pkl")
joblib.dump(target_encoder, "../models/target_encoder.pkl")

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

pd.DataFrame({"interaction_type": y_train}).to_csv(
    "../data/processed/y_train.csv",
    index=False
)

pd.DataFrame({"interaction_type": y_test}).to_csv(
    "../data/processed/y_test.csv",
    index=False
)